# Simple English RAG (`ser`) — GPU Ingestion on Google Colab

This notebook uses Google Colab's free **NVIDIA T4 GPU** to stream, chunk, embed, and ingest the Simple English Wikipedia dump directly into your hosted **Qdrant Cloud** cluster at **~400–600 chunks/second**.

### Instructions:
1. In the menu above, go to **Runtime** > **Change runtime type** and select **T4 GPU**.
2. Fill in your `QDRANT_URL` and `QDRANT_API_KEY` below.
3. Run the cells in order!

In [ ]:
# 1. Clone repository and install dependencies with GPU support
!git clone https://github.com/parm2006/SimpleEnglishRag.git
%cd SimpleEnglishRag
!pip install -q --upgrade pip
!pip install -q -e .
!pip install -q onnxruntime-gpu

# Verify GPU is detected by ONNX Runtime
import onnxruntime as ort
print("Available ONNX Providers:", ort.get_available_providers())
assert "CUDAExecutionProvider" in ort.get_available_providers(), "Please select a GPU runtime (Runtime > Change runtime type > T4 GPU)!"

In [ ]:
# 2. Set your Qdrant Cloud credentials
import os

os.environ["QDRANT_STORAGE"] = "cloud"
os.environ["QDRANT_COLLECTION"] = "simple_wiki"
os.environ["QDRANT_URL"] = "https://139ad1e1-3990-4fef-b123-e75fd7182f21.us-west-1-0.aws.cloud.qdrant.io"
os.environ["QDRANT_API_KEY"] = "YOUR_QDRANT_API_KEY_HERE"  # <-- Replace with your API key

from ser.db import client, COLLECTION_NAME
info = client.get_collection(COLLECTION_NAME)
print(f"Connected to Qdrant Cloud! Currently contains {info.points_count} points.")

In [ ]:
# 3. Download the Wikimedia dump directly into Colab's high-speed cloud disk (~5-10 seconds)
from ser.download import download_dump
dump_file = download_dump(dest_dir="data")
print("Dump ready at:", dump_file)

In [ ]:
# 4. Run High-Speed GPU Ingestion
# This streams the remaining articles and uploads directly to your Qdrant Cloud cluster
import time
from pathlib import Path
from fastembed import TextEmbedding
from ser.dump import iter_dump_articles
from ser.pipeline import index_documents
import ser.embed

# Activate CUDA GPU provider for FastEmbed
ser.embed.model = TextEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    providers=["CUDAExecutionProvider"]
)

dump_path = Path("data/simplewiki-latest-pages-articles.xml.bz2")
checkpoint_path = Path("data/ingest_checkpoint.json")

# Stream articles (resumes automatically from checkpoint or article 2,700)
print("Starting GPU ingestion into Qdrant Cloud...")
articles_stream = iter_dump_articles(
    dump_path,
    checkpoint_path=checkpoint_path,
    reset=False
)

t0 = time.time()
total = index_documents(articles_stream, client=client, batch_size=256)
t1 = time.time()

print(f"\nIngestion Finished! Indexed {total:,} chunks in {(t1-t0)/60:.1f} minutes.")

In [ ]:
# 5. Check Final Qdrant Stats
info = client.get_collection(COLLECTION_NAME)
print(f"Final Qdrant Cloud Collection points: {info.points_count:,}")